In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
orders_data = [
(1,"C1","Chennai","2024-01-01",500),
(2,"C2","Chennai","2024-01-01",700),
(3,"C1","Chennai","2024-01-02",300),
(4,"C3","Madurai","2024-01-01",400),
(5,"C2","Chennai","2024-01-03",900),
(6,"C4","Madurai","2024-01-02",200),
(7,"C3","Madurai","2024-01-03",800),
(8,"C1","Chennai","2024-01-04",1000),
(9,"C5","Salem","2024-01-01",600),
(10,"C5","Salem","2024-01-02",300)
]

df = spark.createDataFrame(orders_data,
["order_id","customer_id","city","order_date","amount"])
display(df)

In [0]:
df_1 = df.withColumn("total",sum("amount").over(Window.partitionBy("customer_id").orderBy("order_date")))
display(df_1)

In [0]:
df_2 = df.withColumn("Cumunitive_sum",sum("amount").over(Window.orderBy("order_date")))
df_2.show()

#USER DEFINE FUNCTION

In [0]:
data = [
    (1,"mani",20),
    (2,"arun",8),
    (3,"kumar",10)
]

df_udf = spark.createDataFrame(data,["id","name","age"])

df_udf.show()

def greet(name):
    return "Hello  " + name

def cap(name):
    return name.upper()

def adult(age):
    if age >= 18:
        return "Adult"
    else:
        return "Minor"
    
adult = udf(adult,StringType())
cap_name = udf(cap,StringType())
greet_name = udf(greet,StringType())

df_udf = df_udf.withColumn("name_upper_version",cap_name("name"))
df_udf = df_udf.withColumn("Greet",greet_name("name"))
df_udf = df_udf.withColumn("age_level",adult("age"))
display(df_udf)

#DELTA

In [0]:
df_udf.write.format("delta")\
    .mode("overwrite")\
    .option("path","/Volumes/sparkcatalog/raw/source/delta")\
    .save()

In [0]:
df=spark.read.format("delta").load("/Volumes/sparkcatalog/raw/source/delta")
display(df)

In [0]:
new_data = [
    (4,"suresh",25),
    (5,"ramesh",30),
    (6,"santhosh",15)
]

df_new = spark.createDataFrame(new_data,["id","name","age"])



df_udf_new = df_new.withColumn("name_upper_version",cap_name("name"))
df_udf_new = df_udf_new.withColumn("Greet",greet_name("name"))
df_udf_new = df_udf_new.withColumn("age_level",adult("age"))
display(df_udf_new)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark,"/Volumes/sparkcatalog/raw/source/delta/")

delta_table.alias("trg").merge(df_udf_new.alias("src"),"trg.id = src.id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

In [0]:
df=spark.read.format("delta").load("/Volumes/sparkcatalog/raw/source/delta")
display(df)

In [0]:
df_json = spark.read.format("json")\
    .load("/Volumes/sparkcatalog/raw/source/json/durty.json")
display(df_json)

In [0]:
delta_table = DeltaTable